# Task 096: holopy_hpiv — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Holographic particle image velocimetry using digital holography

**Paper**: HoloPy: Holography and Light Scattering in Python
**Repository**: https://github.com/manoharan-lab/holopy

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 21.34 dB
- **SSIM**: N/A (3D positions)
- **Evaluation**: 3D particle position recovery from hologram (RMSE + CC)

### Mathematical Formulation

**Digital holography** records interference between reference $R$ and object $O$ waves:

$$I(x,y) = |R+O|^2 = |R|^2 + |O|^2 + R^*O + RO^*$$

**Numerical reconstruction** via Fresnel back-propagation:
$$U(x',y',d) = \mathcal{F}^{-1}\left\{\mathcal{F}\{I \cdot R^*\} \cdot H(f_x,f_y,d)\right\}$$

where $H = \exp\left(i\frac{2\pi d}{\lambda}\sqrt{1-\lambda^2 f_x^2 - \lambda^2 f_y^2}\right)$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python3
"""
Holographic PIV — 3D Particle Position Recovery from Inline Holograms.

Forward model (Angular Spectrum Method):
  H(x,y) = |E_ref + Σ_i E_scat_i|²
  E_scat_i = ASM_propagate(shadow_i, z_i)

Inverse model:
  1. Back-propagate hologram to z-planes via ASM
  2. Local focus metric (gradient-based Tamura coefficient)
  3. Peak detection in 3-D focus volume → (x, y, z)

Metrics: RMSE, Pearson CC, PSNR of 3-D positions.
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`_asm_kernel`, `_shadow`, `detect_particles_3d`, `match_particles`, `_cc`, `make_fig`, `main`

In [ ]:
def _asm_kernel(nx, ny, dx, z, wl):
    fx = np.fft.fftfreq(nx, d=dx)
    fy = np.fft.fftfreq(ny, d=dx)
    FX, FY = np.meshgrid(fx, fy, indexing="ij")
    kz2 = (1.0/wl)**2 - FX**2 - FY**2
    prop = kz2 > 0
    return np.exp(1j*2*np.pi*np.sqrt(np.maximum(kz2, 0))*z) * prop

def _shadow(nx, ny, dx, x0, y0, r):
    xx = np.arange(nx)*dx;  yy = np.arange(ny)*dx
    XX, YY = np.meshgrid(xx, yy, indexing="ij")
    s = np.zeros((nx, ny), dtype=complex)
    s[(XX-x0)**2 + (YY-y0)**2 <= r**2] = -1.0
    return s

def detect_particles_3d(holo, dx, z_planes, wl, n_exp):
    nx, ny = holo.shape;  nz = len(z_planes)
    H_fft = np.fft.fft2(np.sqrt(holo.astype(complex)))
    grad_vol = np.zeros((nz, nx-1, ny-1), dtype=np.float32)
    for i, z in enumerate(z_planes):
        E = np.fft.ifft2(H_fft * _asm_kernel(nx, ny, dx, -z, wl))
        I = np.abs(E)**2
        gx = np.diff(I, axis=0)[:,:-1]
        gy = np.diff(I, axis=1)[:-1,:]
        grad_vol[i] = gaussian_filter(np.sqrt(gx**2+gy**2).astype(np.float32), sigma=3.0)

    mip = np.max(grad_vol, axis=0)
    thr = np.percentile(mip, 93)
    lab, nf = label(mip > thr)
    centroids = center_of_mass(mip, lab, range(1, nf+1))

    det = []
    for cy, cx in centroids:
        iy, ix = int(round(cy)), int(round(cx))
        if 0 <= iy < grad_vol.shape[1] and 0 <= ix < grad_vol.shape[2]:
            zidx = int(np.argmax(grad_vol[:, iy, ix]))
            det.append([(ix+0.5)*dx, (iy+0.5)*dx, z_planes[zidx]])
    return (np.array(det) if det else np.zeros((0,3))), grad_vol

def match_particles(gt, det, md):
    if len(det)==0 or len(gt)==0: return np.zeros((0,3)), np.zeros((0,3))
    D = cdist(gt, det); mg=[]; md_=[]; ug=set(); ud=set()
    for idx in np.argsort(D, axis=None):
        gi, di = np.unravel_index(idx, D.shape)
        if gi in ug or di in ud: continue
        if D[gi,di] > md: break
        mg.append(gt[gi]); md_.append(det[di]); ug.add(gi); ud.add(di)
    return np.array(mg), np.array(md_)

def _cc(a, b):
    af, bf = a.ravel(), b.ravel()
    return float(np.corrcoef(af, bf)[0,1]) if np.std(af)>1e-15 and np.std(bf)>1e-15 else 0.0

def make_fig(holo, gt, det, mg, md, gv, zp, path):
    um=1e6; dx=PIXEL_SIZE
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    ax=axes[0,0]; ext=[0,holo.shape[0]*dx*um,0,holo.shape[1]*dx*um]
    im=ax.imshow(holo.T, cmap="gray", origin="lower", extent=ext)
    ax.set_title("Simulated Inline Hologram"); ax.set_xlabel("x (μm)"); ax.set_ylabel("y (μm)")
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax=axes[0,1]; mip=np.max(gv, axis=0); ext2=[0,mip.shape[0]*dx*um,0,mip.shape[1]*dx*um]
    im=ax.imshow(mip.T, cmap="hot", origin="lower", extent=ext2)
    ax.set_title("Focus MIP (x-y)"); ax.set_xlabel("x (μm)"); ax.set_ylabel("y (μm)")
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax=axes[1,0]
    ax.scatter(gt[:,0]*um, gt[:,1]*um, facecolors="none", edgecolors="blue", s=120, lw=2, label="GT", zorder=2)
    if len(det)>0: ax.scatter(det[:,0]*um, det[:,1]*um, c="red", marker="x", s=80, lw=2, label="Det", zorder=3)
    if len(mg)>0:
        for g,d in zip(mg*um, md*um): ax.plot([g[0],d[0]],[g[1],d[1]],"g--",alpha=.5,lw=.8)
    ax.set_title("Top (x-y)"); ax.set_xlabel("x (μm)"); ax.set_ylabel("y (μm)"); ax.legend(); ax.set_aspect("equal")

    ax=axes[1,1]
    ax.scatter(gt[:,0]*um, gt[:,2]*um, facecolors="none", edgecolors="blue", s=120, lw=2, label="GT", zorder=2)
    if len(det)>0: ax.scatter(det[:,0]*um, det[:,2]*um, c="red", marker="x", s=80, lw=2, label="Det", zorder=3)
    if len(mg)>0:
        for g,d in zip(mg*um, md*um): ax.plot([g[0],d[0]],[g[2],d[2]],"g--",alpha=.5,lw=.8)
    ax.set_title("Side (x-z)"); ax.set_xlabel("x (μm)"); ax.set_ylabel("z (μm)"); ax.legend()

    plt.tight_layout(); plt.savefig(str(path), dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"  Saved → {path}")

def main():
    t0 = time.time()
    print("="*70)
    print("Holographic PIV — 3D Particle Recovery")
    print("="*70)

    print("\n[1/6] Particles …")
    gt_parts = generate_particles(N_PARTICLES, NX, NY, PIXEL_SIZE, Z_MIN, Z_MAX, R_MIN, R_MAX, MARGIN, RNG)
    gt_pos = gt_parts[:,:3]; n_gt = len(gt_parts)
    print(f"  {n_gt} particles  z∈[{gt_pos[:,2].min()*1e6:.0f},{gt_pos[:,2].max()*1e6:.0f}] μm")

    print("\n[2/6] Hologram …")
    holo = simulate_hologram(gt_parts, NX, NY, PIXEL_SIZE, WAVELENGTH)
    print(f"  {holo.shape}  I∈[{holo.min():.4f},{holo.max():.4f}]")

    print("\n[3/6] Backprop ({} z-planes) …".format(Z_SCAN_N))
    zp = np.linspace(Z_SCAN_MIN, Z_SCAN_MAX, Z_SCAN_N)
    det, gv = detect_particles_3d(holo, PIXEL_SIZE, zp, WAVELENGTH, n_gt)
    print(f"  Detected {len(det)}")

    print("\n[4/6] Metrics …")
    mg, md = match_particles(gt_pos, det, MATCH_DIST)
    nm = len(mg)
    if nm > 0:
        r3=_rmse(mg,md); rxy=_rmse(mg[:,:2],md[:,:2]); rz=_rmse(mg[:,2:],md[:,2:])
        cc=_cc(mg,md); p=_psnr(mg,md)
    else:
        r3=rxy=rz=float("inf"); cc=p=0.0
    print(f"  Matched {nm}/{n_gt}  RMSE={r3*1e6:.2f}μm  CC={cc:.4f}  PSNR={p:.2f}dB")

    print("\n[5/6] Save …")
    for d in [WORKING_DIR, ASSET_DIR]:
        d.mkdir(parents=True, exist_ok=True)
        np.save(str(d/"gt_output.npy"), gt_pos)
        np.save(str(d/"recon_output.npy"), det)
    m = dict(n_gt=int(n_gt), n_detected=int(len(det)), n_matched=int(nm),
             detection_rate=round(nm/n_gt,4), rmse_3d_um=round(r3*1e6,2),
             rmse_xy_um=round(rxy*1e6,2), rmse_z_um=round(rz*1e6,2),
             cc=round(cc,4), psnr_db=round(p,2))
    with open(str(WORKING_DIR/"metrics.json"),"w") as f: json.dump(m, f, indent=2)

    print("\n[6/6] Plot …")
    for d in [ASSET_DIR, WORKING_DIR]:
        make_fig(holo, gt_parts, det, mg, md, gv, zp, d/"vis_result.png")

    el = time.time()-t0
    print(f"\n{'='*70}")
    print(f"DONE ({el:.1f}s)  PSNR={p:.2f}dB  CC={cc:.4f}  RMSE={r3*1e6:.2f}μm")
    print("="*70)
    return m

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_particles`, `simulate_hologram`

In [ ]:
def generate_particles(n, nx, ny, dx, zlo, zhi, rlo, rhi, margin, rng):
    pts = []; sep_xy = 25*dx; sep_z = 40e-6
    for _ in range(n*100):
        if len(pts) >= n: break
        x = rng.uniform(margin*dx, (nx-margin)*dx)
        y = rng.uniform(margin*dx, (ny-margin)*dx)
        z = rng.uniform(zlo, zhi)
        r = rng.uniform(rlo, rhi)
        if all(not (abs(x-p[0])<sep_xy and abs(y-p[1])<sep_xy and abs(z-p[2])<sep_z) for p in pts):
            pts.append((x, y, z, r))
    return np.array(pts)

def simulate_hologram(parts, nx, ny, dx, wl):
    E = np.ones((nx, ny), dtype=complex)
    for x0, y0, z0, rad in parts:
        sh = _shadow(nx, ny, dx, x0, y0, rad)
        E += np.fft.ifft2(np.fft.fft2(sh) * _asm_kernel(nx, ny, dx, z0, wl))
    return np.abs(E)**2

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `_rmse`, `_psnr`

In [ ]:
def _rmse(a, b): return float(np.sqrt(np.mean((a-b)**2)))

def _psnr(a, b):
    mse = np.mean((a-b)**2)
    mx = np.max(np.abs(a))
    return 100.0 if mse<1e-30 else (0.0 if mx<1e-30 else float(10*np.log10(mx**2/mse)))

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
t0 = time.time()
print("="*70)
print("Holographic PIV — 3D Particle Recovery")
print("="*70)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("\n[1/6] Particles …")
gt_parts = generate_particles(N_PARTICLES, NX, NY, PIXEL_SIZE, Z_MIN, Z_MAX, R_MIN, R_MAX, MARGIN, RNG)
gt_pos = gt_parts[:,:3]; n_gt = len(gt_parts)
print(f"  {n_gt} particles  z∈[{gt_pos[:,2].min()*1e6:.0f},{gt_pos[:,2].max()*1e6:.0f}] μm")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print("\n[2/6] Hologram …")
holo = simulate_hologram(gt_parts, NX, NY, PIXEL_SIZE, WAVELENGTH)
print(f"  {holo.shape}  I∈[{holo.min():.4f},{holo.max():.4f}]")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("\n[3/6] Backprop ({} z-planes) …".format(Z_SCAN_N))
zp = np.linspace(Z_SCAN_MIN, Z_SCAN_MAX, Z_SCAN_N)
det, gv = detect_particles_3d(holo, PIXEL_SIZE, zp, WAVELENGTH, n_gt)
print(f"  Detected {len(det)}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("\n[4/6] Metrics …")
mg, md = match_particles(gt_pos, det, MATCH_DIST)
nm = len(mg)
if nm > 0:
    r3=_rmse(mg,md); rxy=_rmse(mg[:,:2],md[:,:2]); rz=_rmse(mg[:,2:],md[:,2:])
    cc=_cc(mg,md); p=_psnr(mg,md)
else:
    r3=rxy=rz=float("inf"); cc=p=0.0
print(f"  Matched {nm}/{n_gt}  RMSE={r3*1e6:.2f}μm  CC={cc:.4f}  PSNR={p:.2f}dB")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("\n[5/6] Save …")
for d in [WORKING_DIR, ASSET_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    np.save(str(d/"gt_output.npy"), gt_pos)
    np.save(str(d/"recon_output.npy"), det)
m = dict(n_gt=int(n_gt), n_detected=int(len(det)), n_matched=int(nm),
         detection_rate=round(nm/n_gt,4), rmse_3d_um=round(r3*1e6,2),
         rmse_xy_um=round(rxy*1e6,2), rmse_z_um=round(rz*1e6,2),
         cc=round(cc,4), psnr_db=round(p,2))
with open(str(WORKING_DIR/"metrics.json"),"w") as f: json.dump(m, f, indent=2)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("\n[6/6] Plot …")
for d in [ASSET_DIR, WORKING_DIR]:
    make_fig(holo, gt_parts, det, mg, md, gv, zp, d/"vis_result.png")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
el = time.time()-t0
print(f"\n{'='*70}")
print(f"DONE ({el:.1f}s)  PSNR={p:.2f}dB  CC={cc:.4f}  RMSE={r3*1e6:.2f}μm")
print("="*70)
return m

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **holopy_hpiv**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=21.34 dB, SSIM=N/A (3D positions))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: HoloPy: Holography and Light Scattering in Python
- Repository: https://github.com/manoharan-lab/holopy
